### 1) Installation & Dateien

In [ ]:
!pip install py3Dmol mdtraj -q

# --- Dateien bereitstellen ---
!wget -q "https://raw.githubusercontent.com/robert-scr/QCforQC/main/PUS_new.parm7"
!wget -q "https://raw.githubusercontent.com/robert-scr/QCforQC/main/OPES_production.dcd"
TOP  = "PUS_new.parm7"          # Topologie (parm7/prmtop oder pdb)
TRAJ = "OPES_production.dcd"    # Trajektorie (dcd)
FRAME = 0                       # welcher Frame gezeigt wird


### 2) Trajektorie laden

In [ ]:
import mdtraj as md
import numpy as np

traj = md.load(TRAJ, top=TOP)
top  = traj.topology
print(traj)


### 3) Wichtige Atomgruppen definieren

`serials_small_qm()` im Original erzeugt eine NGL-`@`-Selektion. NGL zählt bei `@`
die **Atom-Indizes 0-basiert** — das entspricht direkt `mdtraj`-Atomindizes.
Falls die Markierung um genau ein Atom verschoben wirkt, `INDEX_OFFSET = -1` setzen.

In [ ]:
def add_serials(sel, a, b):
    # entspricht dem Original: range(a, b) -> a .. b-1
    sel.extend(range(a, b))
    return sel

def qm_indices_small():
    sel = []
    add_serials(sel, 1191, 1197)     # 1191..1196
    add_serials(sel, 10109, 10147)   # 10109..10146
    # --- weitere Gruppen aus dem Original (bei Bedarf einkommentieren): ---
    # add_serials(sel, 1617, 1632)
    # add_serials(sel, 10364, 10365)
    # add_serials(sel, 3139, 3160)
    # add_serials(sel, 57900, 57903)
    # add_serials(sel, 10644, 10647)
    return sel

INDEX_OFFSET = 0
qm = [i + INDEX_OFFSET for i in qm_indices_small()]
qm = [i for i in qm if 0 <= i < top.n_atoms]

# Residuum 173 (im Original: view.add_ball_and_stick('173'))
# NGL nummeriert nach Residuen-Nummer -> in mdtraj 'resSeq'
res173 = list(top.select("resSeq 173"))
if not res173:
    res173 = list(top.select("resid 173"))   # Fallback: 0-basierter Index

# Kontrolle: was ist tatsaechlich markiert?
print(f"QM-Atome: {len(qm)},  Residuum-173-Atome: {len(res173)}\n")
for i in qm:
    a = top.atom(i)
    print(f"  idx {i:>6}  {a.name:<4} {a.residue.name:<4} resSeq {a.residue.resSeq}")


### 4) Rendern

Aufbau als getrennte py3Dmol-Modelle (umgeht die NGL-vs-py3Dmol-Selektionssyntax vollständig):
- Modell 0: Cartoon (Kontext, transparent)
- Modell 1: QM-Atome, cyan-Kohlenstoffe
- Modell 2: Residuum 173, orange-Kohlenstoffe
- gestrichelte Linien: angenäherte Kontakte (polare Atome 2.4–3.6 Å)

In [ ]:
import py3Dmol, itertools

snap = traj[FRAME]

def write_subset(indices, path):
    idx = list(indices)
    if not idx:
        return False
    snap.atom_slice(idx).save_pdb(path)
    return True

macro_idx = top.select("protein or nucleic")
write_subset(macro_idx, "macro.pdb")
write_subset(qm, "qm.pdb")
have173 = write_subset(res173, "res173.pdb")

view = py3Dmol.view(width=900, height=700)
m = 0

# --- Kontext-Cartoon (aus Zelle 14) ---
if len(macro_idx) > 0:
    view.addModel(open("macro.pdb").read(), "pdb")
    view.setStyle({'model': m}, {'cartoon': {'color': 'lightgrey', 'opacity': 0.3}})
    m += 1

# --- QM-Atome: Ball-and-Stick ---
view.addModel(open("qm.pdb").read(), "pdb")
view.setStyle({'model': m}, {'stick':  {'radius': 0.15, 'colorscheme': 'cyanCarbon'},
                             'sphere': {'scale': 0.25,  'colorscheme': 'cyanCarbon'}})
m_qm = m
m += 1

# --- Residuum 173: Ball-and-Stick ---
if have173:
    view.addModel(open("res173.pdb").read(), "pdb")
    view.setStyle({'model': m}, {'stick':  {'radius': 0.15, 'colorscheme': 'orangeCarbon'},
                                 'sphere': {'scale': 0.25,  'colorscheme': 'orangeCarbon'}})
    m += 1

# --- angenaeherte Kontakte (Ersatz fuer add_contact) ---
xyz = snap.xyz[0] * 10.0  # nm -> Angstrom, absolute Koordinaten
polar = [i for i in qm if top.atom(i).element is not None
         and top.atom(i).element.symbol in ('N', 'O', 'F')]
n_contacts = 0
for i, j in itertools.combinations(polar, 2):
    d = float(np.linalg.norm(xyz[i] - xyz[j]))
    if 2.4 <= d <= 3.6:
        a, b = xyz[i], xyz[j]
        view.addLine({'dashed': True, 'color': 'yellow',
                      'start': {'x': float(a[0]), 'y': float(a[1]), 'z': float(a[2])},
                      'end':   {'x': float(b[0]), 'y': float(b[1]), 'z': float(b[2])}})
        n_contacts += 1
print("angenaeherte Kontakte:", n_contacts)

view.zoomTo({'model': m_qm})
view.show()


### 5) Optional: QM-Region als Animation über die Trajektorie

py3Dmol animiert Frames aus einer Multi-Model-PDB. Hier nur die (kleine) QM-Region,
damit es leicht bleibt.

In [ ]:
import py3Dmol

sub = traj.atom_slice(qm)
sub.save_pdb("qm_traj.pdb")            # ein MODEL-Block pro Frame

v = py3Dmol.view(width=900, height=700)
v.addModelsAsFrames(open("qm_traj.pdb").read(), "pdb")
v.setStyle({}, {'stick':  {'radius': 0.15, 'colorscheme': 'cyanCarbon'},
                'sphere': {'scale': 0.25,  'colorscheme': 'cyanCarbon'}})
v.zoomTo()
v.animate({'loop': 'forward', 'interval': 200})
v.show()
